# Federated (GradAvg) — Supervised Fine-Tuning (SFT)

$$\mathcal{L}_{\mathrm{SFT}} = -\frac{1}{|y^{+}|}\sum_{t} \log \pi_{\theta}\left(y^{+}_{t} \mid q, c^{+}, y^{+}_{<t}\right)$$

Token-level cross-entropy on the preferred answer under the relevant snippet.
Prompt tokens are masked, so only answer tokens contribute.

---

*Notation:* `q` query · `c+` relevant snippet · `c-` off-topic snippet from the same user · `y+` personalised answer · `y-` general answer


In [1]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip -q install -U datasets huggingface_hub transformers accelerate peft bitsandbytes rouge-score

    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    DRIVE_ROOT = Path(".")

if IN_COLAB:
    PP_ROOT = DRIVE_ROOT / "privacy_perserving_pllm"
else:
    _here = Path(".").resolve()
    PP_ROOT = next(
        (
            p for p in [_here, *_here.parents]
            if (p / "v2_personamem_persona_subsets").exists()
            or all((p / d).exists() for d in ("centralised", "evaluation", "fed_grad_avg", "zero_shot"))
        ),
        _here,
    )
SUBSET_NAME = "all_subset"
SUBSETS_DIR = PP_ROOT / "v2_personamem_persona_subsets"

RESULTS_DIR = PP_ROOT / SUBSET_NAME / "v2_personamem_federated_sft_gradavg_snippet"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Subset:", SUBSET_NAME)
print("Results will be saved to:", RESULTS_DIR.resolve())

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 207.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 67.3 MB/s eta 0:00:00
Mounted at /content/drive
Subset: all_subset
Results will be saved to: /content/drive/MyDrive/privacy_perserving_pllm/all_subset/v2_personamem_federated_sft_gradavg_snippet


In [2]:
import ast
import json
import random
from typing import Any, Dict, List

from tqdm.auto import tqdm

import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer

DATASET_NAME = "bowen-upenn/PersonaMem-v2"
CONFIG = "benchmark"
TEXT_SPLITS = ("train_text", "val_text", "benchmark_text")

SEED = 42
TOKENIZER_NAME = "Qwen/Qwen3-0.6B"
random.seed(SEED)


def load_split(split):
    if split not in TEXT_SPLITS:
        raise ValueError(f"split must be one of {TEXT_SPLITS}, got {split!r}")
    return load_dataset(DATASET_NAME, CONFIG, split=split)


def parse_user_query(raw):
    if isinstance(raw, dict):
        return str(raw.get("content", raw)).strip()
    if isinstance(raw, str):
        try:
            d = ast.literal_eval(raw)
            if isinstance(d, dict):
                return str(d.get("content", raw)).strip()
        except (ValueError, SyntaxError):
            pass
        return raw.strip()
    return str(raw).strip()


def parse_incorrect_answers(raw):
    if isinstance(raw, list):
        return [str(x) for x in raw]
    if hasattr(raw, "tolist"):
        return [str(x) for x in raw.tolist()]
    if isinstance(raw, str):
        try:
            val = ast.literal_eval(raw)
            if isinstance(val, list):
                return [str(x) for x in val]
        except (ValueError, SyntaxError):
            pass
        return [raw]
    return []

def get_snippet(row):
    val = row.get("related_conversation_snippet")
    if val is None:
        return ""
    return str(val).strip()

In [3]:
def load_subset(name, subsets_dir=SUBSETS_DIR):
    """Load a persona subset (ids + train/val rows) saved by create_persona_subsets.ipynb."""
    subset_dir = Path(subsets_dir) / name
    if not subset_dir.exists():
        raise FileNotFoundError(f"Subset not found: {subset_dir}. Run create_persona_subsets.ipynb first.")
    with open(subset_dir / "personas.json", "r", encoding="utf-8") as f:
        persona_ids = [int(p) for p in json.load(f)]
    tr = pd.read_parquet(subset_dir / "train.parquet")
    va = pd.read_parquet(subset_dir / "val.parquet")
    return sorted(persona_ids), tr, va

MIN_VAL_ROWS = 4
CLIENT_PERSONAS, train_df, val_df = load_subset(SUBSET_NAME)
NUM_CLIENTS = len(CLIENT_PERSONAS)
val_counts = val_df.groupby("persona_id").size()
print(f"train rows: {len(train_df):,} | val rows: {len(val_df):,}")
print(f"Subset '{SUBSET_NAME}': {NUM_CLIENTS} personas")
print(f"Client personas ({len(CLIENT_PERSONAS)}): first 10 = {CLIENT_PERSONAS[:10]} ...")
print(f"Val rows per selected persona (min/mean/max): "
      f"{val_counts[CLIENT_PERSONAS].min()}/{val_counts[CLIENT_PERSONAS].mean():.1f}/{val_counts[CLIENT_PERSONAS].max()}")

train rows: 3,870 | val rows: 714
Subset 'all_subset': 150 personas
Client personas (150): first 10 = [6, 9, 14, 18, 33, 41, 51, 56, 57, 73] ...
Val rows per selected persona (min/mean/max): 4/4.8/8


In [4]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is not available. This notebook needs a GPU.\n"
        "In Google Colab: Runtime → Change runtime type → Hardware accelerator → GPU "
        "(e.g. T4), then Runtime → Restart session and re-run from the top."
    )
print("GPU:", torch.cuda.get_device_name(0))
from datasets import Dataset
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    get_peft_model_state_dict,
    prepare_model_for_kbit_training,
    set_peft_model_state_dict,
)
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, DataCollatorForSeq2Seq

MODEL_NAME = "Qwen/Qwen3-0.6B"
MAX_SEQ_LEN = 4096
MAX_SNIPPET_TOKENS = 2048
MAX_ANSWER_TOKENS = 512
MAX_NEW_TOKENS = 512

START_MODE = "hf"
CONTINUE_FROM_EPOCH = 3
CONTINUE_EPOCHS = 1
GLOBAL_EPOCHS = 3
GLOBAL_LR = 1e-3
GLOBAL_WEIGHT_DECAY = 0.0
MAX_GRAD_NORM = 1.0
LOCAL_BATCH_SIZE = 4

EVAL_BATCH_SIZE = 32
EVAL_TRAIN = False

model_tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if model_tok.pad_token is None:
    model_tok.pad_token = model_tok.eos_token
model_tok.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

collator = DataCollatorForSeq2Seq(model_tok, label_pad_token_id=-100, padding=True, return_tensors="pt")
KEEP_COLS = ["user_query", "correct_answer", "related_conversation_snippet"]

def build_system_prompt():
    return (
        "You are a personalised assistant. Use the conversation snippet to find "
        "information or connections relevant to the question, then provide the answer."
    )

def truncate_snippet(snippet, max_tokens):
    ids = model_tok(snippet, add_special_tokens=False)["input_ids"]
    if len(ids) <= max_tokens:
        return snippet
    return model_tok.decode(ids[-max_tokens:], skip_special_tokens=True)

def build_user_content(row):
    snippet = truncate_snippet(get_snippet(row), MAX_SNIPPET_TOKENS)
    return (
        "Relevant snippet from our earlier conversation:\n"
        f"{snippet}\n\n"
        f"{parse_user_query(row['user_query'])}"
    )

def build_prompt_text(system_prompt, row):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": build_user_content(row)},
    ]
    return model_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def make_tokenize_fn(system_prompt):
    def fn(example):
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": build_user_content(example)},
        ]
        prompt_text = model_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompt_ids = model_tok(prompt_text, add_special_tokens=False)["input_ids"]
        answer_ids = model_tok(
            str(example["correct_answer"]) + model_tok.eos_token,
            add_special_tokens=False, truncation=True, max_length=MAX_ANSWER_TOKENS,
        )["input_ids"]
        input_ids = (prompt_ids + answer_ids)[:MAX_SEQ_LEN]
        labels = ([-100] * len(prompt_ids) + answer_ids)[:MAX_SEQ_LEN]
        return {"input_ids": input_ids, "labels": labels, "attention_mask": [1] * len(input_ids)}
    return fn

client_data: Dict[Any, Dict[str, Any]] = {}
for pid in CLIENT_PERSONAS:
    p_train = train_df[train_df["persona_id"] == pid].reset_index(drop=True)
    system_prompt = build_system_prompt()
    ds = Dataset.from_pandas(p_train[KEEP_COLS].reset_index(drop=True))
    tokenized = ds.map(make_tokenize_fn(system_prompt), remove_columns=ds.column_names)
    examples = tokenized.to_list()
    client_data[pid] = {
        "examples": examples,
        "n": len(examples),
        "system_prompt": system_prompt,
    }
    print(f"  client persona {pid}: {len(examples)} train examples")

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

  client persona 980: 31 train examples


Map:   0%|          | 0/21 [00:00<?, ? examples/s]

  client persona 987: 21 train examples


Map:   0%|          | 0/23 [00:00<?, ? examples/s]

  client persona 996: 23 train examples


In [11]:
def load_continual_adapter(from_epoch):
    """Continual learning: load base model + a previously saved LoRA adapter (trainable)."""
    adapter_dir = RESULTS_DIR / "global" / f"adapter_epoch_{from_epoch}"
    if not adapter_dir.exists():
        raise FileNotFoundError(f"Adapter to continue from not found: {adapter_dir}")
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto",
        trust_remote_code=True, attn_implementation="sdpa",
    )
    base.config.use_cache = False
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
    base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    base.enable_input_require_grads()
    m = PeftModel.from_pretrained(base, str(adapter_dir), is_trainable=True)
    m.print_trainable_parameters()
    return m


def load_base_with_adapter():
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto",
        trust_remote_code=True, attn_implementation="sdpa",
    )
    base.config.use_cache = False
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
    base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    base.enable_input_require_grads()
    m = get_peft_model(base, lora_config)
    return m


def clone_state(model):
    return {k: v.detach().cpu().clone() for k, v in get_peft_model_state_dict(model).items()}


class ClientBatchSampler:
    """Yields fixed-size batches of a client's examples, reshuffling on wrap-around."""

    def __init__(self, rows, batch_size, seed=SEED):
        self.rows = rows
        self.batch_size = batch_size
        self.rng = random.Random(seed)
        self.order: List[int] = []
        self.pos = 0
        self._reshuffle()

    def _reshuffle(self):
        self.order = list(range(len(self.rows)))
        self.rng.shuffle(self.order)
        self.pos = 0

    def new_epoch(self):
        self._reshuffle()

    def next_batch(self):
        if not self.rows:
            return []
        batch = []
        while len(batch) < self.batch_size:
            if self.pos >= len(self.order):
                self._reshuffle()
            batch.append(self.rows[self.order[self.pos]])
            self.pos += 1
        return batch


def sft_batch_loss(model, batch_examples):
    """Mean causal-LM loss over a batch of tokenized SFT examples (keeps graph for backward)."""
    batch = collator(batch_examples)
    device = next(model.parameters()).device
    batch = {k: v.to(device) for k, v in batch.items()}
    if device.type == "cuda":
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            loss = model(**batch).loss
    else:
        loss = model(**batch).loss
    return loss, {"loss": float(loss.detach().cpu())}

In [12]:
import math

if START_MODE == "adapter":
    model = load_continual_adapter(CONTINUE_FROM_EPOCH)
    _start_epoch = CONTINUE_FROM_EPOCH + 1
    _end_epoch = CONTINUE_FROM_EPOCH + CONTINUE_EPOCHS
    print(f"Continual learning from adapter_epoch_{CONTINUE_FROM_EPOCH}: "
          f"training epochs {_start_epoch}..{_end_epoch}")
else:
    model = load_base_with_adapter()
    _start_epoch = 1
    _end_epoch = GLOBAL_EPOCHS
    print(f"Fresh HuggingFace base model: training epochs {_start_epoch}..{_end_epoch}")

model.train()
model.config.use_cache = False

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=GLOBAL_LR, weight_decay=GLOBAL_WEIGHT_DECAY)

active_personas = [pid for pid in CLIENT_PERSONAS if client_data[pid]["examples"]]
num_active = len(active_personas)
samplers = {
    pid: ClientBatchSampler(client_data[pid]["examples"], LOCAL_BATCH_SIZE)
    for pid in active_personas
}

steps_per_epoch = max(
    math.ceil(len(client_data[pid]["examples"]) / LOCAL_BATCH_SIZE)
    for pid in active_personas
)
_num_train_epochs = _end_epoch - _start_epoch + 1
GLOBAL_STEPS = _num_train_epochs * steps_per_epoch
print(
    f"Active clients: {num_active}/{len(CLIENT_PERSONAS)} | steps/epoch: {steps_per_epoch} "
    f"| global steps: {GLOBAL_STEPS}"
)

GLOBAL_DIR = RESULTS_DIR / "global"
GLOBAL_DIR.mkdir(parents=True, exist_ok=True)
epoch_adapter_dirs: List[str] = []
training_log: Dict[str, Any] = {"method": "federated_sft_gradavg_global", "steps": []}

global_state = clone_state(model)
global_step = 0
for epoch in range(_start_epoch, _end_epoch + 1):
    for s in samplers.values():
        s.new_epoch()
    epoch_step_losses: List[float] = []

    for step in tqdm(range(1, steps_per_epoch + 1), desc=f"Epoch {epoch}/{_end_epoch}"):
        set_peft_model_state_dict(model, global_state)
        optimizer.zero_grad(set_to_none=True)
        client_step_losses: List[float] = []

        for pid in active_personas:
            batch = samplers[pid].next_batch()
            batch_loss, _ = sft_batch_loss(model, batch)
            (batch_loss / num_active).backward()
            client_step_losses.append(float(batch_loss.detach().cpu()))

        if MAX_GRAD_NORM:
            torch.nn.utils.clip_grad_norm_(params, MAX_GRAD_NORM)
        optimizer.step()

        global_state = clone_state(model)
        global_step += 1
        mean_step_loss = sum(client_step_losses) / len(client_step_losses)
        epoch_step_losses.append(mean_step_loss)
        training_log["steps"].append({
            "epoch": epoch, "step": step, "global_step": global_step,
            "mean_client_loss": mean_step_loss,
        })

    epoch_dir = GLOBAL_DIR / f"adapter_epoch_{epoch}"
    epoch_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(epoch_dir))
    model_tok.save_pretrained(str(epoch_dir))
    epoch_adapter_dirs.append(str(epoch_dir))
    print(
        f"epoch {epoch}/{_end_epoch} | updates this epoch={steps_per_epoch} "
        f"(global_step={global_step}) | mean_loss={sum(epoch_step_losses) / len(epoch_step_losses):.4f} "
        f"| saved {epoch_dir.name}"
    )

global_state = clone_state(model)

with open(GLOBAL_DIR / "training_losses.json", "w", encoding="utf-8") as f:
    json.dump(training_log, f, indent=2)

global_adapter_dir = GLOBAL_DIR / "adapter"
model.save_pretrained(str(global_adapter_dir))
model_tok.save_pretrained(str(global_adapter_dir))

with open(GLOBAL_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump({
        "method": "federated_sft_gradavg_global",
        "aggregation": "gradient_averaging_fedsgd",
        "single_global_optimizer": True,
        "subset": SUBSET_NAME,
        "client_personas": [int(p) for p in CLIENT_PERSONAS],
        "num_clients": len(CLIENT_PERSONAS),
        "num_active_clients": num_active,
        "min_val_rows": MIN_VAL_ROWS,
                "seed": SEED,
        "start_mode": START_MODE,
        "continue_from_epoch": CONTINUE_FROM_EPOCH if START_MODE == "adapter" else None,
        "continue_epochs": CONTINUE_EPOCHS if START_MODE == "adapter" else None,
        "trained_epoch_range": [int(_start_epoch), int(_end_epoch)],
        "global_epochs": GLOBAL_EPOCHS,
        "steps_per_epoch": steps_per_epoch,
        "global_steps": GLOBAL_STEPS,
        "global_lr": GLOBAL_LR,
        "global_weight_decay": GLOBAL_WEIGHT_DECAY,
        "max_grad_norm": MAX_GRAD_NORM,
        "local_batch_size": LOCAL_BATCH_SIZE,
        "epoch_adapter_dirs": epoch_adapter_dirs,
        "final_adapter_dir": str(global_adapter_dir),
    }, f, indent=2)

del optimizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Saved global federated SFT (gradient-averaging) adapter to:", global_adapter_dir.resolve())

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Fresh HuggingFace base model: training epochs 1..3
Active clients: 150/150 | steps/epoch: 10 | global steps: 30


Epoch 1/3:   0%|          | 0/10 [00:00<?, ?it/s]

epoch 1/3 | updates this epoch=10 (global_step=10) | mean_loss=2.7530 | saved adapter_epoch_1


Epoch 2/3:   0%|          | 0/10 [00:00<?, ?it/s]

epoch 2/3 | updates this epoch=10 (global_step=20) | mean_loss=2.2287 | saved adapter_epoch_2


Epoch 3/3:   0%|          | 0/10 [00:00<?, ?it/s]

epoch 3/3 | updates this epoch=10 (global_step=30) | mean_loss=1.9149 | saved adapter_epoch_3
Saved global federated SFT (gradient-averaging) adapter to: /content/drive/MyDrive/privacy_perserving_pllm/all_subset/v2_personamem_federated_sft_gradavg_snippet/global/adapter


In [10]:
print(batch)

[{'input_ids': [151644, 8948, 198, 2610, 525, 264, 84248, 17847, 13, 5443, 279, 10435, 43065, 311, 1477, 1995, 476, 13234, 9760, 311, 279, 3405, 11, 1221, 3410, 279, 4226, 13, 151645, 198, 151644, 872, 198, 693, 8367, 43065, 504, 1039, 6788, 10435, 510, 58, 4913, 5778, 788, 330, 872, 497, 330, 1796, 788, 330, 8610, 10414, 369, 19429, 10655, 2337, 1293, 4115, 518, 264, 18010, 7521, 2137, 5212, 5778, 788, 330, 77091, 497, 330, 1796, 788, 330, 12549, 498, 3139, 27861, 4722, 1182, 6646, 504, 1293, 4115, 518, 697, 18010, 11, 1430, 42368, 697, 10496, 773, 697, 4722, 1182, 374, 1632, 87113, 11, 323, 1896, 2805, 11259, 476, 41849, 18303, 1449, 220, 18, 15, 4142, 21, 15, 4420, 13, 1446, 2578, 1083, 8760, 504, 264, 2613, 40163, 2257, 44638, 311, 10306, 5810, 34676, 17189, 1189, 2137, 5212, 5778, 788, 330, 872, 497, 330, 1796, 788, 330, 5501, 10667, 429, 358, 614, 27861, 4722, 1182, 6646, 504, 43287, 18010, 975, 1189, 2137, 5212, 5778, 788, 330, 77091, 497, 330, 1796, 788, 330, 32462, 432, 1959, 

In [ ]:
def load_global_model(epoch=None):
    """Load the federated global SFT LoRA adapter. Use epoch=1..GLOBAL_EPOCHS for checkpoints."""
    if epoch is not None:
        adapter_dir = RESULTS_DIR / "global" / f"adapter_epoch_{epoch}"
    else:
        adapter_dir = RESULTS_DIR / "global" / "adapter"
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto",
        trust_remote_code=True, attn_implementation="sdpa",
    )
    base.config.use_cache = True
    m = PeftModel.from_pretrained(base, str(adapter_dir))
    m.eval()
    tok = AutoTokenizer.from_pretrained(str(adapter_dir))
    return m, tok

print("Final global adapter:", RESULTS_DIR / "global" / "adapter")
print("Per-epoch checkpoints:", RESULTS_DIR / "global" / "adapter_epoch_<n>")
print("Per-persona val preds:", RESULTS_DIR / "persona_<id>" / "val_predictions.csv")